# Scoring
> Score a sampling campaign against the ground truth

Work in progress: experiments for scoring campaigns and comparing sampling strategies, following `scoring-plan.md` (#12).

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import wasserstein_distance
import geopandas as gpd
from fastcore.test import *
from fastcore.utils import store_attr

from trufl.utils import gridder, zonal_values
from trufl.sampler import Sampler, rank_to_sample
from trufl.collector import DataCollector
from trufl.callbacks import State, MaxCB, MinCB, StdCB, CountCB, MoranICB, PriorCB
from trufl.optimizer import Optimizer

One campaign to experiment on: a 10 x 10 grid over the simulated raster, with between 0 and 9 samples per unit, so some units have none:

In [ ]:
fname_raster = 'files/ground-truth-01-4326-simulated.tif'
gdf_grid = gridder(fname_raster, nrows=10, ncols=10)
n = np.random.default_rng(0).integers(0, 10, size=len(gdf_grid))
samples = DataCollector(fname_raster).collect(Sampler(gdf_grid).sample(n, method='uniform', rng=0))
samples.head()

`unit_stats` compares each unit's samples with the raster cells inside it. Units without samples have a count of 0 and `NaN` for the measured values and the distance:

In [ ]:
def unit_stats(
    samples:gpd.GeoDataFrame, # Measurements indexed by `loc_id`, with a `value` column
    gdf_units:gpd.GeoDataFrame, # Units indexed by `loc_id`
    fname_raster:str, # Ground-truth raster
) -> pd.DataFrame: # One row per unit
    "True and measured mean and maximum, sample count and Wasserstein distance of each unit."
    rows = {}
    for loc_id, geom in gdf_units.geometry.items():
        true = zonal_values(fname_raster, geom)
        meas = samples.value[samples.index == loc_id].values
        rows[loc_id] = dict(true_mean=true.mean(), true_max=true.max(),
                            meas_mean=meas.mean() if len(meas) else np.nan, meas_max=meas.max() if len(meas) else np.nan,
                            n=len(meas), wasserstein=wasserstein_distance(true, meas) if len(meas) else np.nan)
    return pd.DataFrame.from_dict(rows, orient='index').rename_axis('loc_id')

In [ ]:
df = unit_stats(samples, gdf_grid, fname_raster)
test_eq(df.n.sum(), len(samples))
df.head()

## Campaigns

A `Campaign` simulates a campaign over a ground-truth raster, one round at a time. A round can be a day, a week or a year. `reset` starts a campaign with no samples. `step` takes the number of samples for each unit, draws and measures them, and returns `unit_stats` for all samples so far. `Campaign` decides nothing: the allocation comes from a strategy.

In [ ]:
class Campaign:
    "Simulated campaign over `fname_raster`, sampling the units of `gdf_units` one round at a time."
    def __init__(self,
        fname_raster:str, # Ground-truth raster
        gdf_units:gpd.GeoDataFrame, # Units indexed by `loc_id`
    ):
        store_attr()
        self.sampler, self.collector = Sampler(gdf_units), DataCollector(fname_raster)

    def reset(self,
        seed:int=None, # Seed for sampling and for `MoranICB`'s permutation test
    ) -> gpd.GeoDataFrame: # The samples so far: none
        "Start a new campaign with no samples."
        self.rng, self.rounds = np.random.default_rng(seed), []
        np.random.seed(seed)
        return self.samples

    @property
    def samples(self) -> gpd.GeoDataFrame: # Measurements of every round so far, indexed by `loc_id`, with `value` and `round` columns
        if not self.rounds: return gpd.GeoDataFrame(index=pd.Index([], name='loc_id'), geometry=None, data={'value': None, 'round': None})
        return pd.concat(self.rounds)

    def step(self,
        n:np.ndarray, # Number of samples for each unit, in `gdf_units` order
    ) -> pd.DataFrame: # `unit_stats` of all samples so far
        "Draw and measure `n` samples per unit, then score all samples so far."
        pts = self.sampler.sample(n, method='uniform', rng=self.rng)
        self.rounds.append(self.collector.collect(pts).assign(round=len(self.rounds) + 1))
        return unit_stats(self.samples, self.gdf_units, self.fname_raster)

A strategy is a function that takes the samples so far, the units and the round's budget, and returns the number of samples for each unit. `mcdm` makes the strategy of [Run a campaign over several years](how-to/01_multi_year_campaign.ipynb): rank from the prior alone while there are no samples, then from six state variables, and split the budget with `rank_to_sample`. The prior is the ground truth itself until #11 adds a degraded prior.

In [ ]:
def mcdm(
    fname_prior:str, # Raster used as the prior
    is_benefit:list=[True, True, True, False, False, True], # Benefit or cost for Max, Min, Std, Count, Moran's I and Prior
    weights:list=[0.2, 0.1, 0.1, 0.2, 0.2, 0.2], # Weights of the same six criteria
    policy:str='quantiles', # Allocation policy of `rank_to_sample`
    min:int=1, # Minimum number of samples per unit
):
    "Strategy ranking units by MCDM on the prior alone without samples, then on six state variables."
    def strategy(samples, gdf_units, budget):
        if samples.empty: cbs, isb, w, n_method = [PriorCB(fname_prior)], [True], [1], None
        else:
            cbs = [MaxCB(), MinCB(), StdCB(), CountCB(), MoranICB(k=5), PriorCB(fname_prior)]
            isb, w, n_method = is_benefit, weights, 'LINEAR1'
        state = State(samples, gdf_units, cbs=cbs)()
        ranks = Optimizer(state=state).get_rank(is_benefit_x=isb, w_vector=w, n_method=n_method, c_method=None, w_method=None, s_method='CP')
        return rank_to_sample(ranks['rank'].sort_index().values, budget=budget, min=min, policy=policy)
    return strategy

`run` plays a strategy for one budget per round and returns the per-unit tables of every round, indexed by `round` and `loc_id`:

In [ ]:
def run(
    campaign:Campaign, # Campaign to play
    strategy:callable, # Takes samples so far, units and budget; returns samples per unit
    budgets:list, # Budget of each round
    seed:int=None, # Seed passed to `Campaign.reset`
) -> pd.DataFrame: # `unit_stats` after each round, indexed by `round` and `loc_id`
    "Play `strategy` on `campaign` for `len(budgets)` rounds."
    campaign.reset(seed)
    tables = {r: campaign.step(strategy(campaign.samples, campaign.gdf_units, b)) for r, b in enumerate(budgets, 1)}
    return pd.concat(tables, names=['round'])

Six rounds of 150 samples with the how-to strategy. The same seed gives the same campaign:

In [ ]:
campaign = Campaign(fname_raster, gdf_grid)
res = run(campaign, mcdm(fname_raster), budgets=[150] * 6, seed=0)
test_eq(res.loc[6].n.sum(), len(campaign.samples))
test_eq(campaign.samples['round'].max(), 6)
test_eq(run(campaign, mcdm(fname_raster), budgets=[150] * 6, seed=0), res)

`summarise` averages the errors over units after each round:

In [ ]:
def summarise(res): 
    "Mean over units of the errors in each round of `res`."
    return res.assign(abs_err_mean=(res.meas_mean - res.true_mean).abs(), err_max=res.meas_max - res.true_max
        ).groupby('round').agg(samples=('n', 'sum'), unsampled=('n', lambda n: (n == 0).sum()), mean_abs_err_mean=('abs_err_mean', 'mean'),
                               mean_err_max=('err_max', 'mean'), mean_wasserstein=('wasserstein', 'mean'))

summarise(res).round(4)